CELL 1
# FictionalCart - Customer Reviews Generation

## Microsoft Excel Capstone Project

### Notebook 05: Customer_Reviews_Generation.ipynb

**Project:** FictionalCart – E-commerce Data Analytics

**Purpose:**
Generate a realistic Customer Reviews dataset that integrates with the existing
Customer_Master, Product_Master, Orders, and Returns datasets.

The generated dataset is intended for Microsoft Excel analysis, enabling
business reporting on customer satisfaction, product ratings, review trends,
and the relationship between reviews and product returns.

---

### Input Files

- Customer_Master.csv
- Product_Master.csv
- Orders.csv
- Returns.csv

### Output File

- Customer_Reviews.csv

---

### Business Rules (Frozen)

- Reviews are generated only for Delivered and Returned orders.
- Approximately 25% of eligible orders receive a review.
- One review per eligible order.
- Customer_ID and Product_ID are inherited from Orders.
- Review_Date occurs between 3 and 20 days after Order_Date.
- Ratings range from 1 to 5.
- Returned orders are more likely to receive lower ratings.
- No duplicate Review_ID.
- No duplicate Order_ID.
- No missing values.

---

Author: Devika Reddy Goluguri
Project: Microsoft Excel Data Analyst Portfolio
Company: FictionalCart (Fictional E-commerce Business)

Status: Development

In [1]:
#CELL 2 - IMPORT LIBRARIES

import random
import numpy as np
import pandas as pd

from pathlib import Path
from datetime import timedelta

In [12]:
#CELL 3 - CONFIGURATION

RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

INPUT_FOLDER = Path(".")
OUTPUT_FOLDER = Path(".")

REVIEW_PROBABILITY = 0.25

MIN_REVIEW_DELAY = 3
MAX_REVIEW_DELAY = 20

In [13]:
#CELL 4 - LOAD MASTER DATA

customers_df = pd.read_csv(INPUT_FOLDER / "Customer_Master.csv")
products_df = pd.read_csv(INPUT_FOLDER / "Product_Master.csv")
orders_df = pd.read_csv(INPUT_FOLDER / "Orders.csv")
returns_df = pd.read_csv(INPUT_FOLDER / "Returns.csv")

In [14]:
#CELL 5 - QUICK DATASET VERIFICATION

print("=" * 60)
print("INPUT DATASET SUMMARY")
print("=" * 60)

print(f"Customers : {customers_df.shape}")
print(f"Products  : {products_df.shape}")
print(f"Orders    : {orders_df.shape}")
print(f"Returns   : {returns_df.shape}")

INPUT DATASET SUMMARY
Customers : (750, 8)
Products  : (200, 6)
Orders    : (5000, 7)
Returns   : (253, 7)


In [15]:
#CELL 6 - CHECK REQUIRED COLUMNS

required_order_columns = [
    "Order_ID",
    "Order_Date",
    "Customer_ID",
    "Product_ID",
    "Order_Amount",
    "Order_Status"
]

missing_columns = [
    col
    for col in required_order_columns
    if col not in orders_df.columns
]

if missing_columns:
    raise ValueError(f"Missing columns in Orders.csv: {missing_columns}")

print("✅ Orders.csv contains all required columns.")

✅ Orders.csv contains all required columns.


In [16]:
#CELL 7 - FILTER ELIGIBLE ORDERS

eligible_orders = orders_df[
    orders_df["Order_Status"].isin(["Delivered", "Returned"])
].copy()

print("=" * 60)
print("ELIGIBLE ORDERS")
print("=" * 60)

print(f"Eligible Orders : {len(eligible_orders):,}")

ELIGIBLE ORDERS
Eligible Orders : 4,622


In [17]:
#CELL 8 - DETERMINE NUMBER OF REVIEWS

review_count = round(len(eligible_orders) * REVIEW_PROBABILITY)

print("=" * 60)
print("REVIEW GENERATION PLAN")
print("=" * 60)

print(f"Eligible Orders      : {len(eligible_orders):,}")
print(f"Review Probability   : {REVIEW_PROBABILITY:.0%}")
print(f"Reviews To Generate  : {review_count:,}")

REVIEW GENERATION PLAN
Eligible Orders      : 4,622
Review Probability   : 25%
Reviews To Generate  : 1,156


In [18]:
#CELL 9 - GENERATE REVIEW IDs

def generate_review_ids(n):
    """
    Generate sequential Review IDs.

    Example:
    R000001
    R000002
    ...
    """

    return [
        f"R{i:06d}"
        for i in range(1, n + 1)
    ]

In [26]:
#CELL 10 - GENERATE REVIEW DATE

PROJECT_END_DATE = pd.Timestamp("2025-12-31")

def generate_review_date(order_date):
    """
    Generate a review date between
    3 and 20 days after the order date,
    without exceeding the project end date.
    """

    order_date = pd.to_datetime(order_date)

    delay = random.randint(
        MIN_REVIEW_DELAY,
        MAX_REVIEW_DELAY
    )

    review_date = order_date + timedelta(days=delay)

    return min(review_date, PROJECT_END_DATE)

In [27]:
#CELL 11 - GENERATE RATING

def generate_rating(order_status):
    """
    Generate a realistic customer rating.

    Returned orders are more likely
    to receive lower ratings.
    """

    if order_status == "Returned":

        ratings = [1, 2, 3, 4, 5]
        probabilities = [0.30, 0.30, 0.20, 0.15, 0.05]

    else:

        ratings = [1, 2, 3, 4, 5]
        probabilities = [0.05, 0.08, 0.15, 0.30, 0.42]

    return np.random.choice(
        ratings,
        p=probabilities
    )

In [28]:
#CELL 12 - RANDOMLY SELECT ORDERS FOR REVIEW

review_orders = eligible_orders.sample(
    n=review_count,
    random_state=RANDOM_SEED
).copy()

review_orders.reset_index(
    drop=True,
    inplace=True
)

print("=" * 60)
print("SELECTED REVIEW ORDERS")
print("=" * 60)

print(f"Selected Reviews : {len(review_orders):,}")

SELECTED REVIEW ORDERS
Selected Reviews : 1,156


In [29]:
#CELL 13 - GENERATE CUSTOMER REVIEWS DATASET

customer_reviews_df = pd.DataFrame({
    "Review_ID": generate_review_ids(len(review_orders)),
    "Order_ID": review_orders["Order_ID"].values,
    "Customer_ID": review_orders["Customer_ID"].values,
    "Product_ID": review_orders["Product_ID"].values,
    "Review_Date": [
        generate_review_date(order_date)
        for order_date in review_orders["Order_Date"]
    ],
    "Rating": [
        generate_rating(status)
        for status in review_orders["Order_Status"]
    ],
    "Order_Status": review_orders["Order_Status"].values
})

print(customer_reviews_df.head())

  Review_ID Order_ID Customer_ID Product_ID Review_Date  Rating Order_Status
0   R000001  O000773       C0099      P0021  2025-12-02       3    Delivered
1   R000002  O000534       C0086      P0061  2024-10-08       5    Delivered
2   R000003  O001706       C0365      P0180  2023-03-28       5    Delivered
3   R000004  O000932       C0374      P0020  2025-10-24       1    Delivered
4   R000005  O002310       C0279      P0048  2024-12-31       5    Delivered


In [30]:
#CELL 14 - VERIFY DATASET STRUCTURE

print("=" * 60)
print("CUSTOMER REVIEWS PREVIEW")
print("=" * 60)

print(customer_reviews_df.head())

print("\nShape")
print(customer_reviews_df.shape)

CUSTOMER REVIEWS PREVIEW
  Review_ID Order_ID Customer_ID Product_ID Review_Date  Rating Order_Status
0   R000001  O000773       C0099      P0021  2025-12-02       3    Delivered
1   R000002  O000534       C0086      P0061  2024-10-08       5    Delivered
2   R000003  O001706       C0365      P0180  2023-03-28       5    Delivered
3   R000004  O000932       C0374      P0020  2025-10-24       1    Delivered
4   R000005  O002310       C0279      P0048  2024-12-31       5    Delivered

Shape
(1156, 7)


In [31]:
#CELL 15 - INITIAL VALIDATION

print("=" * 60)
print("INITIAL VALIDATION")
print("=" * 60)

print(f"Duplicate Review_ID : {customer_reviews_df['Review_ID'].duplicated().sum()}")

print(f"Duplicate Order_ID  : {customer_reviews_df['Order_ID'].duplicated().sum()}")

print(f"Missing Values      : {customer_reviews_df.isna().sum().sum()}")

INITIAL VALIDATION
Duplicate Review_ID : 0
Duplicate Order_ID  : 0
Missing Values      : 0


In [32]:
#CELL 16 - VALIDATE REVIEW DATES
# ==========================================================
# Review Date Validation
# ==========================================================

project_end_date = pd.Timestamp("2025-12-31")

invalid_review_dates = customer_reviews_df[
    customer_reviews_df["Review_Date"] > project_end_date
]

print("=" * 60)
print("REVIEW DATE VALIDATION")
print("=" * 60)

print(f"Reviews After Project End Date : {len(invalid_review_dates)}")

REVIEW DATE VALIDATION
Reviews After Project End Date : 0


In [33]:
#CELL 17 - RATING DISTRIBUTION VALIDATION

print("=" * 60)
print("RATING DISTRIBUTION")
print("=" * 60)

rating_counts = (
    customer_reviews_df["Rating"]
    .value_counts()
    .sort_index()
)

rating_percent = (
    customer_reviews_df["Rating"]
    .value_counts(normalize=True)
    .sort_index()
    * 100
).round(2)

rating_summary = pd.DataFrame({
    "Count": rating_counts,
    "Percentage": rating_percent
})

print(rating_summary)

RATING DISTRIBUTION
        Count  Percentage
Rating                   
1          84        7.27
2         113        9.78
3         167       14.45
4         317       27.42
5         475       41.09


In [34]:
#CELL 18 - COMPARE RATINGS BY ORDER STATUS

print("=" * 60)
print("AVERAGE RATING BY ORDER STATUS")
print("=" * 60)

print(
    customer_reviews_df
    .groupby("Order_Status")["Rating"]
    .mean()
    .round(2)
)

AVERAGE RATING BY ORDER STATUS
Order_Status
Delivered    3.96
Returned     2.23
Name: Rating, dtype: float64


In [35]:
#CELL 19 - REFERENTIAL INTEGRITY VALIDATION

print("=" * 60)
print("REFERENTIAL INTEGRITY")
print("=" * 60)

customer_check = customer_reviews_df["Customer_ID"].isin(
    customers_df["Customer_ID"]
).all()

product_check = customer_reviews_df["Product_ID"].isin(
    products_df["Product_ID"]
).all()

order_check = customer_reviews_df["Order_ID"].isin(
    orders_df["Order_ID"]
).all()

print(f"Customer_ID Valid : {customer_check}")
print(f"Product_ID Valid  : {product_check}")
print(f"Order_ID Valid    : {order_check}")

REFERENTIAL INTEGRITY
Customer_ID Valid : True
Product_ID Valid  : True
Order_ID Valid    : True


In [36]:
#CELL 20 - REMOVE TEMPORARY COLUMN

customer_reviews_df.drop(
    columns=["Order_Status"],
    inplace=True
)

print(customer_reviews_df.columns.tolist())

['Review_ID', 'Order_ID', 'Customer_ID', 'Product_ID', 'Review_Date', 'Rating']


In [37]:
#CELL 21 - EXPORT DATASET

output_file = OUTPUT_FOLDER / "Customer_Reviews.csv"

customer_reviews_df.to_csv(
    output_file,
    index=False
)

print("=" * 60)
print("EXPORT COMPLETE")
print("=" * 60)

print(f"File Name : {output_file.name}")
print(f"Rows      : {len(customer_reviews_df):,}")
print(f"Columns   : {customer_reviews_df.shape[1]}")

EXPORT COMPLETE
File Name : Customer_Reviews.csv
Rows      : 1,156
Columns   : 6


In [38]:
#CELL 22 - FINAL FREEZE SUMMARY

print("=" * 60)
print("NOTEBOOK 05 FREEZE SUMMARY")
print("=" * 60)

print("Notebook : Customer_Reviews_Generation.ipynb")
print("Status   : FROZEN")
print(f"Rows     : {len(customer_reviews_df):,}")
print(f"Columns  : {customer_reviews_df.shape[1]}")

print("\nOutput File")
print("-----------")
print("Customer_Reviews.csv")

print("\nBusiness Rules")
print("--------------")
print("✓ Reviews generated only for Delivered and Returned orders")
print("✓ One review per order")
print("✓ Customer_ID inherited from Orders")
print("✓ Product_ID inherited from Orders")
print("✓ Review dates within project timeline")
print("✓ Ratings generated using weighted probabilities")
print("✓ Referential integrity maintained")
print("✓ No duplicate Review_ID")
print("✓ No duplicate Order_ID")
print("✓ No missing values")

NOTEBOOK 05 FREEZE SUMMARY
Notebook : Customer_Reviews_Generation.ipynb
Status   : FROZEN
Rows     : 1,156
Columns  : 6

Output File
-----------
Customer_Reviews.csv

Business Rules
--------------
✓ Reviews generated only for Delivered and Returned orders
✓ One review per order
✓ Customer_ID inherited from Orders
✓ Product_ID inherited from Orders
✓ Review dates within project timeline
✓ Ratings generated using weighted probabilities
✓ Referential integrity maintained
✓ No duplicate Review_ID
✓ No duplicate Order_ID
✓ No missing values
